# Tome 7, Chapter 1: Banach Spaces (Python companion)

Exercises first, solutions at the end. Infinite-dimensional geometry on a computer: sparse recovery by $\ell^1$ minimisation, Lebesgue constants and divergent Fourier series, weak convergence, and separating hyperplanes as support vector machines. Requirements: numpy, scipy and matplotlib.

*Before each solution, predict what the output will look like.*

## Exercises

**P1.1 (★★) CS application: sparse recovery by $\ell^1$ minimisation.** Let $A$ be a random $m\times n$ Gaussian matrix with $n=200$, and $x_0\in\mathbb R^n$ a vector with $k=10$ non-zero entries. From $b=Ax_0$ with $m<n$ equations, recover $x_0$ by (a) the minimum $\ell^2$-norm solution and (b) the minimum $\ell^1$-norm solution (basis pursuit, a linear program). Plot the success rate of (b) as a function of $m$. Why does the $\ell^1$ ball favour sparse vectors, and what has this to do with Hahn–Banach?

**P1.2 (★) Lebesgue constants and a divergent Fourier series.** Compute $L_N=\frac1{2\pi}\int_{-\pi}^{\pi}|D_N|$ for $N$ up to $10^4$ and compare with $\frac4{\pi^2}\ln N$. Then, for $N=200$, build a continuous $f$ with $|f|\le1$ and $S_Nf(0)$ close to $L_N$ (Theorem 1.9), and plot $f$ with the partial sums $S_nf(0)$, $n\le N$.

**P1.3 (★) Weak convergence on a computer.** For $f_n=\operatorname{sgn}\sin(nx)$ on $[0,2\pi]$, compute $\int f_ng$ for $g(x)=x^{-1/2}$, $g=\mathbf 1_{[0,1]}$ and a random step function, as $n$ grows. Check that $\|f_n\|_\infty=\|f_n\|_2/\sqrt{2\pi}=1$ while $\int f_ng\to0$. Then illustrate Mazur's lemma (Exercise 1.5) in $\ell^2$: the averages $\frac1n(e_1+\dots+e_n)$ and, for a random $\pm$ sequence, $\frac1n\sum\pm e_k$.

**P1.4 (★★) CS application: separating hyperplanes and support vector machines.** Generate two point clouds in the plane whose convex hulls are disjoint. (a) Find a separating line by linear programming (Theorem 1.6). (b) Compute the distance $\delta$ between the two convex hulls by quadratic optimisation, and the maximum-margin separating line. Check that the best margin is $\delta/2$: the minimum distance problem and the maximum margin problem are dual to each other.

## Solutions

### P1.1

In [ ]:
import numpy as np
from scipy.optimize import linprog
rng = np.random.default_rng(0)
n, k = 200, 10

def basis_pursuit(A, b):
    # min sum(u) subject to -u <= x <= u, Ax = b ; variables (x, u)
    m, n = A.shape
    c = np.concatenate([np.zeros(n), np.ones(n)])
    I = np.eye(n)
    A_ub = np.block([[I, -I], [-I, -I]])
    A_eq = np.hstack([A, np.zeros((m, n))])
    res = linprog(c, A_ub=A_ub, b_ub=np.zeros(2 * n), A_eq=A_eq, b_eq=b, bounds=[(None, None)] * n + [(0, None)] * n)
    return res.x[:n]

def trial(m):
    A = rng.standard_normal((m, n)) / np.sqrt(m)
    x0 = np.zeros(n); x0[rng.choice(n, k, replace=False)] = rng.standard_normal(k)
    b = A @ x0
    x2 = np.linalg.pinv(A) @ b
    x1 = basis_pursuit(A, b)
    return np.linalg.norm(x2 - x0) / np.linalg.norm(x0), np.linalg.norm(x1 - x0) / np.linalg.norm(x0)

e2, e1 = trial(60)
print(f"m=60: relative error, min l2 norm: {e2:.3f}   min l1 norm: {e1:.2e}")
ms = list(range(10, 81, 5))
rate = [np.mean([trial(m)[1] < 1e-4 for _ in range(20)]) for m in ms]
for m, r in zip(ms, rate):
    print(f"m={m:3d}: l1 recovery succeeds in {100 * r:5.1f}% of trials")

With $m=60$ equations for $200$ unknowns, the minimum $\ell^2$-norm solution is useless (its error is of the order of $x_0$ itself: it spreads the energy over all coordinates), while basis pursuit recovers $x_0$ to machine precision. The success rate jumps from $0$ to $1$ over a short range of $m$ (here between about $m=30$ and $m=55$, for $k=10$): a sharp phase transition, near $m\approx2k\ln(n/k)\approx60$ in the worst case and lower on average, as Donoho and Tanner showed. Geometrically, the solutions of $Ax=b$ form an affine subspace, and the smallest $\ell^1$ ball that touches it touches it at a vertex or a low-dimensional face, where most coordinates vanish. The $\ell^2$ ball is round and touches anywhere. The certificate that the $\ell^1$ minimiser is $x_0$ is a *dual* vector: a linear form $y^TA$ that equals $\operatorname{sgn}x_0$ on the support and has modulus less than $1$ elsewhere, a supporting hyperplane of the $\ell^1$ ball at $x_0$ (Theorem 1.6). Compressed sensing (Candès, Romberg, Tao, Donoho, 2006) is built on this, and it is why an MRI scan can be taken from far fewer measurements than pixels.

### P1.2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def lebesgue_constant(N, M=200_000):
    t = (np.arange(M) + 0.5) * np.pi / M                    # midpoint rule on ]0, pi[, D_N is even
    D = np.sin((N + 0.5) * t) / np.sin(t / 2)
    return np.mean(np.abs(D))                                # (1/2pi) * 2 * integral over ]0, pi[ = mean
for N in [1, 10, 100, 1000, 10_000]:
    L = lebesgue_constant(N)
    print(f"N={N:6d}: L_N = {L:.5f}   (4/pi^2) ln N = {4 / np.pi ** 2 * np.log(N):.5f}   difference {L - 4 / np.pi ** 2 * np.log(N):.5f}")

N = 200
x = np.linspace(-np.pi, np.pi, 40001)
D = np.sin((N + 0.5) * x) / np.sin(x / 2 + 1e-300)
f = np.clip(50 * D, -1, 1)                                 # continuous, |f| <= 1, close to sgn D_N
coef = lambda n: np.trapezoid(f * np.exp(-1j * n * x), x) / (2 * np.pi)
c = np.array([coef(n) for n in range(-N, N + 1)])
partial = [np.real(c[N - n:N + n + 1].sum()) for n in range(N + 1)]
print(f"S_N f(0) = {partial[-1]:.4f}, L_N = {lebesgue_constant(N):.4f}, while max|f| = {np.abs(f).max()}")
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot(x, f, lw=0.5); ax[0].set_xlim(-0.3, 0.3); ax[0].set_title('f near 0 (zoom)')
ax[1].plot(partial); ax[1].set_title('partial sums $S_n f(0)$'); plt.show()

The constants grow exactly like $\frac4{\pi^2}\ln N$, the difference converging to about $1.27$ (the true constant is $\frac4{\pi^2}(\gamma+\ln 4+2\sum_{k\ge1}\frac{\ln k}{4k^2-1})\approx1.2706$). The function $f$, a clipped Dirichlet kernel, is continuous with $|f|\le1$, and its partial sum $S_{200}f(0)$ reaches about $L_{200}\approx3.4$, more than three times $\|f\|_\infty$. For a single $f$ this is harmless, since $S_nf(0)$ for $n\ne200$ behaves differently; Theorem 1.9 glues infinitely many such bumps, at frequencies growing fast enough, into one continuous function whose partial sums at $0$ are unbounded. The growth is only logarithmic, which is why divergence is so hard to see in practice, and why Fejér's averages, whose kernel has $L^1$ norm $1$, never suffer from it.

### P1.3

In [ ]:
import numpy as np
x = np.linspace(0, 2 * np.pi, 2_000_001)[1:]
dx = x[1] - x[0]
rng = np.random.default_rng(2)
edges = np.sort(rng.uniform(0, 2 * np.pi, 9)); vals = rng.standard_normal(10)
step = vals[np.searchsorted(edges, x)]
gs = {"x^(-1/2)": x ** -0.5, "1_[0,1]": (x <= 1).astype(float), "random step": step}
for n in [1, 10, 100, 1000, 10_000]:
    fn = np.sign(np.sin(n * x))
    row = "   ".join(f"{name}: {np.sum(fn * g) * dx:+.5f}" for name, g in gs.items())
    print(f"n={n:6d}  ||f_n||_2^2/(2 pi) = {np.sum(fn ** 2) * dx / (2 * np.pi):.3f}   {row}")
for n in [10, 100, 1000, 10_000]:
    signs = rng.choice([-1, 1], n)
    print(f"n={n:6d}: ||(e_1+...+e_n)/n|| = {1 / np.sqrt(n):.4f}   ||(sum of random +-e_k)/n|| = {np.linalg.norm(signs) / n:.4f}")

The norms stay at $1$ while the integrals against each weight go to $0$: for the indicator and the step function at rate $\frac1n$ (only the incomplete half-periods at the jumps count, Section 1.5), and for the unbounded $x^{-1/2}$ more slowly, roughly like $\frac1{\sqrt n}$, because most of its mass sits near $0$ within the first half-period. Weak* convergence has no uniform rate: the speed depends on the test function, which is why the weak* topology has no norm in infinite dimension. Mazur's averages, on the other hand, converge in norm, at rate $\frac1{\sqrt n}$ whatever the signs: orthogonal vectors average out like independent random variables, which is the law of large numbers in disguise (Tome 3).

### P1.4

In [ ]:
import numpy as np
from scipy.optimize import linprog, minimize
import matplotlib.pyplot as plt
rng = np.random.default_rng(3)
P = rng.normal([0, 0], 0.6, (40, 2)); Q = rng.normal([3, 2], 0.6, (40, 2))

# (a) feasibility LP: find (w, c) with w.p <= c - 1 on P and w.q >= c + 1 on Q
A_ub = np.vstack([np.hstack([P, -np.ones((40, 1))]), np.hstack([-Q, np.ones((40, 1))])])
res = linprog(np.zeros(3), A_ub=A_ub, b_ub=-np.ones(80), bounds=[(None, None)] * 3)
w_lp, c_lp = res.x[:2], res.x[2]
print("LP separating line found:", res.status == 0, " w =", np.round(w_lp, 3), " c =", round(c_lp, 3))

# (b) distance between the convex hulls: min |P^T a - Q^T b| over probability vectors a, b
def obj(z): return np.sum((z[:40] @ P - z[40:] @ Q) ** 2)
cons = [{'type': 'eq', 'fun': lambda z: z[:40].sum() - 1}, {'type': 'eq', 'fun': lambda z: z[40:].sum() - 1}]
z = minimize(obj, np.full(80, 1 / 40), bounds=[(0, 1)] * 80, constraints=cons, method='SLSQP', options={'maxiter': 500}).x
p, q = z[:40] @ P, z[40:] @ Q
delta = np.linalg.norm(p - q)
w = (q - p) / delta; c = w @ (p + q) / 2                    # the perpendicular bisector of [p, q]
margin = min(np.min(c - P @ w), np.min(Q @ w - c))
print(f"distance between hulls = {delta:.4f}, margin of the bisector = {margin:.4f}, delta/2 = {delta / 2:.4f}")
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(*P.T, s=10); ax.scatter(*Q.T, s=10); ax.plot(*np.array([p, q]).T, 'k.-')
t = np.linspace(-2, 5, 2); ax.plot(t, (c - w[0] * t) / w[1], 'k'); ax.set_aspect('equal'); ax.set_ylim(-2, 4); plt.show()

The linear program finds *a* separating line, as Theorem 1.6 promises for disjoint compact convex sets. The quadratic program finds the closest points $p\in\operatorname{conv}P$ and $q\in\operatorname{conv}Q$, and the perpendicular bisector of $[p,q]$ separates with margin exactly $\frac\delta2$ (equal to the computed margin up to the optimiser's tolerance). No line can do better: a line with margin $r$ puts a strip of width $2r$ between the hulls, so $2r\le\delta$. The proof of Theorem 1.6(b) was this argument: fatten both sets by $r$ until just before they touch. A support vector machine computes exactly this maximum-margin hyperplane (after a change of variables), and the points of $P$ and $Q$ that touch the margin, the *support vectors*, are the only ones that matter: moving any other point slightly does not change the answer. Chapter 2 will add kernels, which separate clouds that no line can separate.